
# Occupancy Prediction


> **Requirements:**  
> - Files must include a `date` column parsable by pandas.  
> - Defaults expect the datasets at `data/datatraining.txt`, `data/datatest.txt`, and `data/datatest2.txt` (CSV-compatible).  
> - You can edit the file paths in the **Parameters** cell.


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from typing import Tuple

In [3]:

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(r'[^\w]+', '_', regex=True)
          .str.replace(r'_+', '_', regex=True)
          .str.strip('_')
    )
    return df

def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = normalize_columns(df)
    if "date" not in df.columns:
        raise ValueError(f"Expected a 'date' column in {path}. Found: {list(df.columns)}")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    return df

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["hour"] = out["date"].dt.hour
    out["dow"] = out["date"].dt.dayofweek
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    
    out["is_weekend"] = out["dow"].isin([5, 6]).astype(int)
    out["is_work_hours"] = out["hour"].between(8, 18).astype(int)
    return out

def split_X_y(df: pd.DataFrame, target: str = "occupancy") -> Tuple[pd.DataFrame, pd.Series]:
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found. Columns: {list(df.columns)}")
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [c for c in numeric_cols if c != target]
    X = df[feature_cols]
    y = df[target].astype(int)
    return X, y


In [ ]:
def build_model():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=5000, solver="liblinear"))
    ])
    
    param_grid = {
        "logreg__C": [0.1, 1.0, 3.0],
        "logreg__class_weight": [None, "balanced"],
    }
    return pipe, param_grid

## Parameters

In [5]:

# Edit these paths to match your data locations.
train_path = Path("data/datatraining.txt")
test1_path = Path("data/datatest.txt")
test2_path = Path("data/datatest2.txt")

# Output directory for any artifacts (e.g., processed CSVs)
outdir = Path("./out_logreg")
outdir.mkdir(parents=True, exist_ok=True)

target = "occupancy"


## Load data, engineer features, and split X/y

In [6]:

# Load
train = load_data(train_path)
test1 = load_data(test1_path)
test2 = load_data(test2_path)

# Feature engineering
train_fe = add_time_features(train)
test1_fe = add_time_features(test1)
test2_fe = add_time_features(test2)

# Split into features/target
X_train, y_train = split_X_y(train_fe, target=target)
X_test1, y_test1 = split_X_y(test1_fe, target=target)
X_test2, y_test2 = split_X_y(test2_fe, target=target)

# Show heads
display(X_train.head())
display(y_train.head())


,temperature,humidity,light,co2,humidityratio,hour,dow,hour_sin,hour_cos,is_weekend,is_work_hours
0,23.18,27.2720,426.0,721.25,0.004793,17,2,-0.965926,-0.258819,0,1
1,23.15,27.2675,429.5,714.00,0.004783,17,2,-0.965926,-0.258819,0,1
2,23.15,27.2450,426.0,713.50,0.004779,17,2,-0.965926,-0.258819,0,1
3,23.15,27.2000,426.0,708.25,0.004772,17,2,-0.965926,-0.258819,0,1
4,23.10,27.2000,426.0,704.50,0.004757,17,2,-0.965926,-0.258819,0,1


0    1
1    1
2    1
3    1
4    1
Name: occupancy, dtype: int64

In [ ]:
pipe, param_grid = build_model()
tscv = TimeSeriesSplit(n_splits=4)
grid = GridSearchCV(
    pipe, param_grid=param_grid, scoring="f1", cv=tscv, n_jobs=1, refit=True, verbose=0
)


## (Optional) Save processed data

In [ ]:

# Uncomment to save processed datasets for reuse
# train_fe.to_csv(outdir / "train_features.csv", index=False)
# test1_fe.to_csv(outdir / "test1_features.csv", index=False)
# test2_fe.to_csv(outdir / "test2_features.csv", index=False)

# X_train.to_csv(outdir / "X_train.csv", index=False)
# y_train.to_csv(outdir / "y_train.csv", index=False)
# X_test1.to_csv(outdir / "X_test1.csv", index=False)
# y_test1.to_csv(outdir / "y_test1.csv", index=False)
# X_test2.to_csv(outdir / "X_test2.csv", index=False)
# y_test2.to_csv(outdir / "y_test2.csv", index=False)
